# Notebook 4: Full Training Loop Pipeline

**Goal:** Demonstrate the complete VRDFormer training pipeline — forward pass, loss computation, backward pass, evaluation, and inference in a simplified, educational setting.

We'll build a minimal training loop for both Stage 1 (pair detection + tracking) and Stage 2 (relation classification), then show evaluation and inference.

## 1. Setup & DDP Workaround

The codebase assumes DDP wrapping (`model.module.xxx`). For notebooks, we create a simple wrapper to avoid errors.

In [1]:
import os
import sys
import json
import time
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from argparse import Namespace
from tqdm import tqdm
import matplotlib.pyplot as plt

# Add repo root
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

import util.misc as utils
from util.misc import NestedTensor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

Device: cpu
PyTorch: 2.13.0+cpu


In [2]:
# DDP Wrapper: Makes model accessible via .module like DistributedDataParallel does
class DDPWrapper(nn.Module):
    """Simple wrapper so model.module.xxx works without DDP."""
    def __init__(self, model):
        super().__init__()
        self.module = model
    
    def forward(self, *args, **kwargs):
        return self.module(*args, **kwargs)
    
    def __getattr__(self, name):
        if name == 'module':
            return self._modules['module']
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self.module, name)

print('DDPWrapper created for notebook compatibility')

DDPWrapper created for notebook compatibility


## 2. Stage 1 Training: Detection + Tracking

### 2a. Build Model, Dataset, Optimizer

In [3]:
from models import build_model
from datasets import build_dataset
from torch.utils.data import DataLoader

# Build Stage 1 args
def get_stage1_args():
    args = Namespace()
    args.dataset = 'vidvrd'
    args.stage = 1
    args.device = str(device)
    args.backbone = 'resnet101'
    args.lr_backbone = 1e-5
    args.dilation = False
    args.position_embedding = 'sine'
    args.enc_layers = 6
    args.dec_layers = 6
    args.hidden_dim = 256
    args.dim_feedforward = 2048
    args.nheads = 8
    args.dropout = 0.1
    args.num_queries = 100
    args.pre_norm = False
    args.aux_loss = True
    args.deformable = False
    args.num_feature_levels = 1
    args.with_box_refine = False
    args.overflow_boxes = False
    args.multi_frame_attention = False
    args.multi_frame_encoding = False
    args.merge_frame_features = False
    args.multi_frame_attention_separate_encoder = False
    args.focal_loss = True
    args.focal_alpha = 0.25
    args.focal_gamma = 2
    args.obj_loss_coef = 1
    args.verb_loss_coef = 1
    args.bbox_loss_coef = 5
    args.giou_loss_coef = 2
    args.eos_coef = 0.1
    args.set_cost_class = 1
    args.set_cost_bbox = 5
    args.set_cost_giou = 2
    args.set_cost_sub_class = 0.5
    args.set_cost_obj_class = 0.5
    args.set_cost_verb_class = 1
    args.tracking = True
    args.track_attention = False
    args.track_query_false_positive_prob = 0.1
    args.track_query_false_negative_prob = 0.4
    args.track_query_false_positive_eos_weight = True
    args.track_backprop_prev_frame = False
    args.max_duration = 24
    args.seq_len = 2
    args.resolution = 'large'
    args.num_workers = 0
    args.batch_size = 2
    args.distributed = False
    args.gpu = 0
    args.debug = True
    args.cautious = True
    args.by_ratio = False
    args.track_prev_frame_range = 8
    args.track_prev_frame_rnd_augs = 0.01
    args.track_prev_prev_frame = False
    args.track_query_noise = 0.0
    args.lr = 5e-5
    args.optimizer = 'adamw'
    args.weight_decay = 1e-4
    args.lr_drop = 5
    args.schedule = 'linear_with_warmup'
    args.fraction_warmup_steps = 0.01
    args.ema = False
    args.ema_decay = 0.9998
    args.dec_n_points = 4
    args.enc_n_points = 4
    args.output_dir = ''
    args.vis_and_log_interval = 10
    args.eval_skip = 1
    args.clip_max_norm = 0.1
    args.accumulate_steps = 1
    # Paths - ADJUST THESE
    args.vidvrd_path = os.environ.get('VIDVRD_PATH', '/home/zhengsipeng/data/vidvrd')
    args.vidor_path = os.environ.get('VIDOR_PATH', '/home/zhengsipeng/data/vidor')
    args.coco_path = ''
    return args

args_s1 = get_stage1_args()

# Build model
print('Building Stage 1 model...')
model_s1, criterion_s1, weight_dict_s1 = build_model(args_s1)
model_s1 = DDPWrapper(model_s1)  # Wrap for notebook compatibility
model_s1.to(device)

n_params = sum(p.numel() for p in model_s1.parameters() if p.requires_grad)
print(f'Model: {type(model_s1.module).__name__}')
print(f'Trainable params: {n_params/1e6:.2f}M')

Building Stage 1 model...


/mnt/c/Users/Samuel Oliveira/Desktop/CS/VRDFormer_VRD/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/mnt/c/Users/Samuel Oliveira/Desktop/CS/VRDFormer_VRD/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model: VRDFormerTracking
Trainable params: 60.43M


In [4]:
# Build dataset and dataloader
try:
    print('Building Stage 1 dataset...')
    dataset_train_s1 = build_dataset('train', args_s1)
    print(f'Dataset size: {len(dataset_train_s1)} training clips')
    
    loader_train_s1 = DataLoader(
        dataset_train_s1,
        batch_size=args_s1.batch_size,
        shuffle=True,
        collate_fn=utils.collate_fn,
        num_workers=0
    )
    print(f'Dataloader: {len(loader_train_s1)} batches')
    DATASET_AVAILABLE = True
except Exception as e:
    print(f'Dataset not available: {e}')
    print('Will use dummy data for demonstration.')
    DATASET_AVAILABLE = False

Building Stage 1 dataset...


Dataset not available: provided VidVRD path /home/zhengsipeng/data/vidvrd does not exist
Will use dummy data for demonstration.


In [5]:
# Optimizer with differential learning rates (same as optim_initializer)
from util.optim import optim_initializer

optimizer_s1 = optim_initializer(args_s1, model_s1.module)

print('=== OPTIMIZER ===')
print(f'Type: {type(optimizer_s1).__name__}')
for i, pg in enumerate(optimizer_s1.param_groups):
    print(f'Group {i}: lr={pg["lr"]:.2e}, weight_decay={pg["weight_decay"]}, params={len(pg["params"])}')

=== OPTIMIZER ===
Type: AdamW
Group 0: lr=5.00e-05, weight_decay=0.0001, params=206
Group 1: lr=1.00e-05, weight_decay=0.0001, params=93


### 2b. Single Training Step (Stage 1)

Let's trace through ONE training step to understand the loss computation.

In [6]:
print('=== STAGE 1: SINGLE TRAINING STEP ===')
print()

model_s1.train()
criterion_s1.train()

if DATASET_AVAILABLE:
    # Get a real batch
    samples, targets = next(iter(loader_train_s1))
else:
    # Create dummy data
    B = 2
    dummy_img = torch.randn(B, 3, 400, 600)
    samples = NestedTensor.from_tensor_list([dummy_img[i] for i in range(B)])
    targets = [{
        'sub_boxes': torch.rand(3, 4), 'obj_boxes': torch.rand(3, 4),
        'sub_labels': torch.randint(0, 35, (3,)), 'obj_labels': torch.randint(0, 35, (3,)),
        'verb_labels': torch.randint(0, 2, (3, 132)).float(),
        'so_track_ids': torch.tensor([[1,2],[3,4],[5,6]]),
        'sub_track_ids': torch.tensor([1,3,5]), 'obj_track_ids': torch.tensor([2,4,6]),
        'sub_area': torch.rand(3), 'obj_area': torch.rand(3),
        'orig_size': torch.tensor([400, 600]), 'size': torch.tensor([400, 600]),
        'num_inst': 3,
        'prev_image': torch.randn(3, 400, 600),
        'prev_target': {
            'sub_boxes': torch.rand(2, 4), 'obj_boxes': torch.rand(2, 4),
            'sub_labels': torch.randint(0, 35, (2,)), 'obj_labels': torch.randint(0, 35, (2,)),
            'verb_labels': torch.randint(0, 2, (2, 132)).float(),
            'so_track_ids': torch.tensor([[1,2],[7,8]]),
            'sub_track_ids': torch.tensor([1,7]), 'obj_track_ids': torch.tensor([2,8]),
            'sub_area': torch.rand(2), 'obj_area': torch.rand(2),
            'num_inst': 2,
        }
    } for _ in range(B)]

samples = samples.to(device)
targets = [{k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]

print(f'Samples: {tuple(samples.tensors.shape)}')
print(f'Targets: {len(targets)} items, each has {len(targets[0])} keys')
print(f'  sub_boxes: {tuple(targets[0]["sub_boxes"].shape)}')
print(f'  sub_labels: {tuple(targets[0]["sub_labels"].shape)}')

=== STAGE 1: SINGLE TRAINING STEP ===

Samples: (2, 3, 400, 600)
Targets: 2 items, each has 15 keys
  sub_boxes: (3, 4)
  sub_labels: (3,)


In [7]:
# FORWARD PASS
print('=== FORWARD PASS ===')
optimizer_s1.zero_grad()

outputs, targets_out, *_ = model_s1(samples, targets)

# Inspect outputs
print(f'\nOutput keys: {list(outputs.keys())}')
for key in ['pred_sub_logits', 'pred_obj_logits', 'pred_verb_logits', 'pred_sub_boxes', 'pred_obj_boxes', 'hs_embed']:
    if key in outputs:
        print(f'  {key}: {tuple(outputs[key].shape)}')
if 'aux_outputs' in outputs:
    print(f'  aux_outputs: {len(outputs["aux_outputs"])} intermediate outputs')

=== FORWARD PASS ===



Output keys: ['pred_sub_logits', 'pred_obj_logits', 'pred_verb_logits', 'pred_sub_boxes', 'pred_obj_boxes', 'hs_embed', 'aux_outputs']
  pred_sub_logits: (2, 101, 35)
  pred_obj_logits: (2, 101, 35)
  pred_verb_logits: (2, 101, 132)
  pred_sub_boxes: (2, 101, 4)
  pred_obj_boxes: (2, 101, 4)
  hs_embed: (2, 101, 256)
  aux_outputs: 5 intermediate outputs


In [8]:
# LOSS COMPUTATION
print('=== LOSS COMPUTATION ===')

loss_dict = criterion_s1(outputs, targets_out)

print(f'\nIndividual losses:')
for k, v in sorted(loss_dict.items()):
    if v.numel() == 1 and not k.startswith('aux'):
        if 'error' in k or 'acc' in k:
            print(f'  {k:<35} {v.item():>8.2f}%')
        else:
            print(f'  {k:<35} {v.item():>8.4f}')

# Compute weighted total loss
total_loss = sum(loss_dict[k] * weight_dict_s1[k] for k in loss_dict if k in weight_dict_s1)
print(f'\n{"Weighted total loss":<35} {total_loss.item():>8.4f}')
print(f'  = loss_ce * {weight_dict_s1["loss_ce"]} + loss_ce_verb * {weight_dict_s1["loss_ce_verb"]} '
      f'+ loss_bbox * {weight_dict_s1["loss_bbox"]} + loss_giou * {weight_dict_s1["loss_giou"]}')

=== LOSS COMPUTATION ===

Individual losses:
  loss_bbox                             1.2557
  loss_bbox_0                           1.2557
  loss_bbox_1                           1.2557
  loss_bbox_2                           1.2557
  loss_bbox_3                           1.2557
  loss_bbox_4                           1.2557
  loss_ce                               1.0820
  loss_ce_0                             1.0290
  loss_ce_1                             1.0331
  loss_ce_2                             1.0183
  loss_ce_3                             1.0449
  loss_ce_4                             1.0659
  loss_ce_verb                         76.6368
  loss_ce_verb_0                       75.8624
  loss_ce_verb_1                       76.7569
  loss_ce_verb_2                       76.7648
  loss_ce_verb_3                       75.9083
  loss_ce_verb_4                       75.6924
  loss_giou                             1.3719
  loss_giou_0                           1.3719
  loss_giou_1  

In [9]:
# BACKWARD PASS
print('=== BACKWARD PASS ===')

total_loss.backward()

# Check gradient stats
grad_norms = []
for name, p in model_s1.named_parameters():
    if p.grad is not None:
        grad_norms.append(p.grad.norm().item())

print(f'Parameters with gradients: {len(grad_norms)}')
print(f'Mean grad norm: {np.mean(grad_norms):.6f}')
print(f'Max grad norm: {np.max(grad_norms):.6f}')
print(f'Min grad norm: {np.min(grad_norms):.6f}')

# Gradient clipping
torch.nn.utils.clip_grad_norm_(model_s1.parameters(), args_s1.clip_max_norm)

# Optimizer step
optimizer_s1.step()
print(f'\nOptimizer step complete. LR = {optimizer_s1.param_groups[0]["lr"]:.2e}')

=== BACKWARD PASS ===


Parameters with gradients: 299
Mean grad norm: 25.282709
Max grad norm: 325.377502
Min grad norm: 0.000000



Optimizer step complete. LR = 5.00e-05


### 2c. Loss Function Details

VRDFormer uses **focal loss** for classification and **L1 + GIoU** for bounding boxes.

In [10]:
print('=== LOSS FUNCTION DETAILS ===')
print()
print('1. Classification Loss (focal):')
print('   - Applied to sub_class and obj_class predictions')
print('   - Focal loss: FL(p_t) = -alpha * (1-p_t)^gamma * log(p_t)')
print(f'   - alpha={args_s1.focal_alpha}, gamma={args_s1.focal_gamma}')
print('   - class_weight includes eos_coef for no-object class')
print()
print('2. Verb Classification Loss (focal, multi-label):')
print('   - Sigmoid focal loss per verb class (binary)')
print('   - Multi-label: each query can predict MULTIPLE verbs simultaneously')
print(f'   - 132 verb classes (VidVRD)')
print()
print('3. Bounding Box Loss (L1 + GIoU):')
print('   - L1: |pred_boxes - gt_boxes|, averaged over subject AND object')
print('   - GIoU: 1 - GIoU(pred, gt), averaged over subject AND object')
print('   - Both boxes: cxcywh format, normalized to [0,1]')
print()
print('4. Cardinality Loss (logging only):')
print('   - |num_predicted_objects - num_gt_objects|')
print()
print('Loss weights from config:')
for k, v in weight_dict_s1.items():
    print(f'  {k}: {v}')

=== LOSS FUNCTION DETAILS ===

1. Classification Loss (focal):
   - Applied to sub_class and obj_class predictions
   - Focal loss: FL(p_t) = -alpha * (1-p_t)^gamma * log(p_t)
   - alpha=0.25, gamma=2
   - class_weight includes eos_coef for no-object class

2. Verb Classification Loss (focal, multi-label):
   - Sigmoid focal loss per verb class (binary)
   - Multi-label: each query can predict MULTIPLE verbs simultaneously
   - 132 verb classes (VidVRD)

3. Bounding Box Loss (L1 + GIoU):
   - L1: |pred_boxes - gt_boxes|, averaged over subject AND object
   - GIoU: 1 - GIoU(pred, gt), averaged over subject AND object
   - Both boxes: cxcywh format, normalized to [0,1]

4. Cardinality Loss (logging only):
   - |num_predicted_objects - num_gt_objects|

Loss weights from config:
  loss_ce: 1
  loss_ce_verb: 1
  loss_bbox: 5
  loss_giou: 2
  loss_ce_0: 1
  loss_ce_verb_0: 1
  loss_bbox_0: 5
  loss_giou_0: 2
  loss_ce_1: 1
  loss_ce_verb_1: 1
  loss_bbox_1: 5
  loss_giou_1: 2
  loss_ce_2: 1


## 3. Stage 2 Training: Relation Classification

Stage 2 processes clips frame-by-frame, accumulating relation embeddings, then classifies at the end.

In [11]:
try:
    # Build Stage 2 model
    args_s2 = Namespace(**{k:v for k,v in vars(args_s1).items()})
    args_s2.stage = 2
    args_s2.tracking = False
    args_s2.batch_size = 1
    args_s2.seq_len = 8

    model_s2, criterion_s2, weight_dict_s2 = build_model(args_s2)
    model_s2 = DDPWrapper(model_s2)
    model_s2.to(device)

    print(f'Stage 2 model: {type(model_s2.module).__name__}')
    n_params_s2 = sum(p.numel() for p in model_s2.parameters() if p.requires_grad)
    print(f'Trainable params: {n_params_s2/1e6:.2f}M')
    STAGE2_AVAILABLE = True
except Exception as e:
    print(f'Stage 2 model requires CUDA for ROI Align (got: {e})')
    print('Skipping Stage 2 execution on CPU. Use a CUDA-enabled device.')
    STAGE2_AVAILABLE = False

# Stage 2 forward pass requires CUDA (hardcoded .cuda() in roi_align)
if device.type == 'cpu':
    STAGE2_AVAILABLE = False
    print('Stage 2 requires CUDA. Skipping on CPU.')

Stage 2 model: VRDFormer_S2
Trainable params: 60.56M
Stage 2 requires CUDA. Skipping on CPU.


In [12]:
# Simulate a Stage 2 training step
print('=== STAGE 2: SIMULATED TRAINING STEP ===')
print()
print('Stage 2 processes frames sequentially, accumulating memory:')
print()

if DATASET_AVAILABLE:
    try:
        dataset_train_s2 = build_dataset('train', args_s2)
        loader_s2 = DataLoader(dataset_train_s2, batch_size=1, shuffle=True,
                               collate_fn=utils.collate_fn, num_workers=0)
        samples_s2, targets_s2 = next(iter(loader_s2))
        samples_s2 = samples_s2.to(device)
        targets_s2 = [[{k: v.to(device) if isinstance(v, torch.Tensor) else v 
                        for k, v in t.items()} for t in targets_s2[0]]]
        HAVE_S2_DATA = True
    except Exception as e:
        print(f'Stage 2 dataset not available: {e}')
        HAVE_S2_DATA = False
else:
    HAVE_S2_DATA = False

if not STAGE2_AVAILABLE:
    print('Stage 2 not available on CPU - skipping.')
elif not HAVE_S2_DATA:
    print('Using dummy Stage 2 data for illustration.')
    # Create dummy 8-frame clip
    seq_len = 8
    dummy_clip = torch.randn(seq_len, 3, 300, 400)
    samples_s2 = NestedTensor(dummy_clip, torch.zeros(seq_len, 300, 400, dtype=torch.bool))
    samples_s2 = samples_s2.to(device)
    
    # Create per-frame dummy targets with track IDs
    targets_s2 = [[{
        'sub_boxes': torch.rand(2, 4), 'obj_boxes': torch.rand(2, 4),
        'sub_labels': torch.tensor([0, 1]), 'obj_labels': torch.tensor([2, 3]),
        'verb_labels': torch.randint(0, 2, (2, 132)).float(),
        'unscaled_sub_boxes': torch.rand(2, 4) * 400,
        'unscaled_obj_boxes': torch.rand(2, 4) * 400,
        'so_track_ids': torch.tensor([[1, 2], [3, 4]]),
        'sub_track_ids': torch.tensor([1, 3]), 'obj_track_ids': torch.tensor([2, 4]),
        'orig_size': torch.tensor([300, 400]), 'size': torch.tensor([300, 400]),
        'num_inst': 2, 'inst_ids': torch.tensor([0, 1]),
        'video_id': 'dummy_video', 'frame_id': fid,
    } for fid in range(seq_len)]]

=== STAGE 2: SIMULATED TRAINING STEP ===

Stage 2 processes frames sequentially, accumulating memory:

Stage 2 not available on CPU - skipping.


In [13]:
# Stage 2 forward pass (frame by frame)
if STAGE2_AVAILABLE:
    model_s2.train()
    criterion_s2.train()

    print('=== STAGE 2 FORWARD LOOP ===')
    print(f'Processing {args_s2.seq_len} frames sequentially...')
    print()

    memory = None
    for fid in range(min(4, args_s2.seq_len)):  # Show first 4 frames
        cur_frame = samples_s2.select_frame(fid)
        target_f = targets_s2[0][fid]
        is_eos = (fid + 1) == args_s2.seq_len
        memory = model_s2(cur_frame, target_f, memory, eos=is_eos)
        print(f'Frame {fid}: cur_frame={tuple(cur_frame.tensors.shape)}')
        if memory is not None:
            print(f'  Memory keys after frame {fid}: {list(memory.keys())}')
else:
    print('=== STAGE 2 FORWARD LOOP ===')
    print('Skipped: Stage 2 requires CUDA (hardcoded .cuda() in ROI Align).')


=== STAGE 2 FORWARD LOOP ===
Skipped: Stage 2 requires CUDA (hardcoded .cuda() in ROI Align).


In [14]:
# At EOS, compute loss from the accumulated memory
print('=== STAGE 2 LOSS (at end of sequence) ===')
print()
if STAGE2_AVAILABLE and memory is not None:
    loss_dict_s2 = criterion_s2(memory)
    print(f'Loss components:')
    for k, v in sorted(loss_dict_s2.items()):
        if v.numel() == 1:
            print(f'  {k:<35} {v.item():>8.4f}')
    total_s2 = sum(loss_dict_s2[k] * weight_dict_s2[k] for k in loss_dict_s2 if k in weight_dict_s2)
    print(f'\nWeighted total loss: {total_s2.item():>8.4f}')
else:
    print('Skipped: Stage 2 not available on CPU or memory is None.')


=== STAGE 2 LOSS (at end of sequence) ===

Skipped: Stage 2 not available on CPU or memory is None.


## 4. Evaluation

VRDFormer uses standard VRD metrics: detection mAP, recall@K, and tagging precision@K.

In [15]:
print('=== VRD EVALUATION METRICS ===')
print()
print('1. RELATION DETECTION:')
print('   - Measures ability to detect <subject, predicate, object> triplets')
print('   - Uses volumetric IoU (vIoU) >= 0.5 for subject+object trajectories')
print('   - Reports mean Average Precision (mAP) per predicate class')
print('   - Also recall@50 and recall@100 (fraction of GT triplets found in top-K)')
print()
print('2. RELATION TAGGING:')
print('   - Simplified: only checks if correct predicate is assigned')
print('   - No temporal alignment required')
print('   - Reports precision@1, @5, @10')
print()
print('3. ZERO-SHOT & GENERALIZED ZERO-SHOT:')
print('   - Zero-shot: triplets NOT seen in training')
print('   - Generalized zero-shot: evaluates ZS predictions without filtering seen triplets')
print('   - Zero-shot triplets = val_triplets - train_triplets')
print()
print('Evaluation function: util/evaluate.py:evaluate(groundtruth, prediction, dataset)')
print('  Returns: OrderedDict with mAP, rec@50, rec@100, pre@1, pre@5, pre@10')
print('  For overall, zero-shot, and generalized zero-shot settings')

=== VRD EVALUATION METRICS ===

1. RELATION DETECTION:
   - Measures ability to detect <subject, predicate, object> triplets
   - Uses volumetric IoU (vIoU) >= 0.5 for subject+object trajectories
   - Reports mean Average Precision (mAP) per predicate class
   - Also recall@50 and recall@100 (fraction of GT triplets found in top-K)

2. RELATION TAGGING:
   - Simplified: only checks if correct predicate is assigned
   - No temporal alignment required
   - Reports precision@1, @5, @10

3. ZERO-SHOT & GENERALIZED ZERO-SHOT:
   - Zero-shot: triplets NOT seen in training
   - Generalized zero-shot: evaluates ZS predictions without filtering seen triplets
   - Zero-shot triplets = val_triplets - train_triplets

Evaluation function: util/evaluate.py:evaluate(groundtruth, prediction, dataset)
  Returns: OrderedDict with mAP, rec@50, rec@100, pre@1, pre@5, pre@10
  For overall, zero-shot, and generalized zero-shot settings


In [16]:
# Show how evaluation is structured (conceptual, requires real data)
print('=== EVALUATION FLOW (Conceptual) ===')
print()
print('For each video in validation set:')
print('  1. Process all frames through Stage 2 model')
print('  2. At end: relation_classifier(memory, gt=groundtruth, is_eval=True)')
print('     -> For each GT triplet, predict verb from pooled temporal features')
print('  3. Store predictions: {"video_id": [{"triplet": (S,P,O), "score": s}, ...]}')
print('  4. Store groundtruth: {"video_id": [{"triplet": (S,P,O), ...}, ...]}')
print()
print('After all videos:')
print('  evaluate(groundtruth, prediction, dataset)')
print('    -> eval_detection_scores(gt, pred, viou_threshold=0.5)')
print('    -> eval_tagging_scores(gt, pred)')
print('    -> Compute AP via voc_ap(), recall@K, precision@K')
print('    -> Repeat for overall, zero-shot, generalized-zero-shot')

=== EVALUATION FLOW (Conceptual) ===

For each video in validation set:
  1. Process all frames through Stage 2 model
  2. At end: relation_classifier(memory, gt=groundtruth, is_eval=True)
     -> For each GT triplet, predict verb from pooled temporal features
  3. Store predictions: {"video_id": [{"triplet": (S,P,O), "score": s}, ...]}
  4. Store groundtruth: {"video_id": [{"triplet": (S,P,O), ...}, ...]}

After all videos:
  evaluate(groundtruth, prediction, dataset)
    -> eval_detection_scores(gt, pred, viou_threshold=0.5)
    -> eval_tagging_scores(gt, pred)
    -> Compute AP via voc_ap(), recall@K, precision@K
    -> Repeat for overall, zero-shot, generalized-zero-shot


## 5. Simplified Training Loop

A minimal training loop that runs a few iterations on real or dummy data.

In [17]:
def train_one_epoch_simple(model, criterion, loader, optimizer, device, max_batches=10):
    """Simplified training loop for Stage 1."""
    model.train()
    criterion.train()
    
    total_loss = 0.0
    n_batches = 0
    
    weight_dict = criterion.weight_dict
    
    for i, (samples, targets) in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        
        samples = samples.to(device)
        targets = [{k: v.to(device) if isinstance(v, torch.Tensor) else v 
                    for k, v in t.items()} for t in targets]
        
        optimizer.zero_grad()
        
        # Forward
        outputs, targets_out, *_ = model(samples, targets)
        
        # Loss
        loss_dict = criterion(outputs, targets_out)
        losses = sum(loss_dict[k] * weight_dict[k] 
                     for k in loss_dict if k in weight_dict)
        
        # Check for NaN
        if torch.isnan(losses):
            print(f'  [WARNING] NaN loss at batch {i}, skipping')
            continue
        
        # Backward
        losses.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()
        
        total_loss += losses.item()
        n_batches += 1
        
        if i % 5 == 0:
            print(f'  Batch {i}: loss={losses.item():.4f}')
    
    return total_loss / max(1, n_batches)

print('Training function defined.')

Training function defined.


In [18]:
# Run mini training on Stage 1
print('=== MINI TRAINING (Stage 1) ===')
print()

if DATASET_AVAILABLE:
    num_epochs = 2
    max_batches_per_epoch = 5  # Small for demo
    
    history = {'train_loss': []}
    
    for epoch in range(num_epochs):
        avg_loss = train_one_epoch_simple(
            model_s1, criterion_s1, loader_train_s1,
            optimizer_s1, device,
            max_batches=max_batches_per_epoch
        )
        history['train_loss'].append(avg_loss)
        print(f'Epoch {epoch+1}/{num_epochs}: avg_loss = {avg_loss:.4f}')
    
    # Plot loss
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(range(1, num_epochs+1), history['train_loss'], 'b-o')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Average Loss')
    ax.set_title('VRDFormer Stage 1 - Training Loss')
    ax.grid(True, alpha=0.3)
    plt.show()
else:
    print('Skipping: real dataset not available.')
    print('Replace paths in args to point to your VidVRD/VidOR data to run training.')

=== MINI TRAINING (Stage 1) ===

Skipping: real dataset not available.
Replace paths in args to point to your VidVRD/VidOR data to run training.


## 6. Checkpoint Save/Load

VRDFormer uses `util/checkpoints.py` for loading pretrained DETR weights and saving training state.

In [19]:
from util.checkpoints import save_on_master

# Save a checkpoint
checkpoint = {
    'model': model_s1.module.state_dict(),
    'epoch': 0,
    'args': args_s1,
}

ckpt_dir = ROOT_DIR / 'data' / 'ckpts' / 'notebook_demo'
ckpt_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = ckpt_dir / 'checkpoint.pth'

torch.save(checkpoint, ckpt_path)
print(f'Saved checkpoint to {ckpt_path}')
print(f'Checkpoint size: {ckpt_path.stat().st_size / 1e6:.1f} MB')

Saved checkpoint to /mnt/c/Users/Samuel Oliveira/Desktop/CS/VRDFormer_VRD/data/ckpts/notebook_demo/checkpoint.pth
Checkpoint size: 243.7 MB


In [20]:
# Load a checkpoint
loaded = torch.load(ckpt_path, map_location=device, weights_only=False)
print(f'Loaded checkpoint:')
print(f'  Epoch: {loaded["epoch"]}')
print(f'  Model keys: {len(loaded["model"])}')
print(f'  First 5 keys: {list(loaded["model"].keys())[:5]}')

Loaded checkpoint:
  Epoch: 0
  Model keys: 816
  First 5 keys: ['transformer.level_embed', 'transformer.encoder.layers.0.self_attn.in_proj_weight', 'transformer.encoder.layers.0.self_attn.in_proj_bias', 'transformer.encoder.layers.0.self_attn.out_proj.weight', 'transformer.encoder.layers.0.self_attn.out_proj.bias']


## 7. Inference Demo

Run inference on a single frame/video to see predicted relation triplets.

In [21]:
print('=== INFERENCE DEMO (Stage 1) ===')

model_s1.eval()

# Run on a single frame (calling VRDFormer directly, bypassing TrackingBase)
# This avoids the need for prev_target in TrackingBase.forward
with torch.no_grad():
    test_img = torch.randn(1, 3, 400, 600).to(device)
    test_nt = NestedTensor(test_img, torch.zeros(1, 400, 600, dtype=torch.bool).to(device))
    
    # Call VRDFormer.forward directly (bypasses TrackingBase which requires prev_target)
    from models.vrdformer import VRDFormer
    outputs, targets_out, features_all, memory, hs = VRDFormer.forward(model_s1.module, test_nt, targets=None)
    
    # Get predictions
    sub_logits = outputs['pred_sub_logits']  # [1, 100, 36]
    obj_logits = outputs['pred_obj_logits']
    verb_logits = outputs['pred_verb_logits']  # [1, 100, 132]
    sub_boxes = outputs['pred_sub_boxes']  # [1, 100, 4]
    obj_boxes = outputs['pred_obj_boxes']
    
    # Get top-K predictions per query
    sub_probs = F.softmax(sub_logits, dim=-1)
    obj_probs = F.softmax(obj_logits, dim=-1)
    verb_probs = torch.sigmoid(verb_logits)  # sigmoid for multi-label
    
    # Filter: keep queries where both sub and obj are NOT no-object
    no_obj_class = sub_logits.shape[-1] - 1  # last class is no-object
    sub_conf, sub_cls = sub_probs[0].max(dim=-1)  # [100]
    obj_conf, obj_cls = obj_probs[0].max(dim=-1)
    
    object_mask = (sub_cls != no_obj_class) & (obj_cls != no_obj_class)
    
    # Get verb predictions for valid queries
    valid_verb_probs = verb_probs[0, object_mask]  # [M, 132]
    valid_verb_scores, valid_verb_cls = valid_verb_probs.topk(3, dim=-1)  # top-3 verbs per query
    
    n_valid = object_mask.sum().item()
    print(f'\nDetected {n_valid} subject-object pairs (out of 100 queries)')
    print(f'\nTop predictions:')
    for i in range(min(5, n_valid)):
        qi = torch.where(object_mask)[0][i].item()
        print(f'  Query {qi:3d}: '
              f'sub_cls={sub_cls[qi].item():2d} (p={sub_conf[qi].item():.3f}), '
              f'obj_cls={obj_cls[qi].item():2d} (p={obj_conf[qi].item():.3f}), '
              f'sub_box=[{sub_boxes[0,qi,0].item():.3f},{sub_boxes[0,qi,1].item():.3f},'
              f'{sub_boxes[0,qi,2].item():.3f},{sub_boxes[0,qi,3].item():.3f}], '
              f'obj_box=[{obj_boxes[0,qi,0].item():.3f},{obj_boxes[0,qi,1].item():.3f},'
              f'{obj_boxes[0,qi,2].item():.3f},{obj_boxes[0,qi,3].item():.3f}]')
        print(f'         verbs: {[(valid_verb_cls[i,j].item(), valid_verb_scores[i,j].item()) for j in range(3)]}')


=== INFERENCE DEMO (Stage 1) ===



Detected 100 subject-object pairs (out of 100 queries)

Top predictions:
  Query   0: sub_cls= 4 (p=0.096), obj_cls=26 (p=0.065), sub_box=[0.500,0.500,0.119,0.119], obj_box=[0.500,0.500,0.119,0.119]
         verbs: [(92, 0.05544959381222725), (128, 0.04233197495341301), (53, 0.03922457620501518)]
  Query   1: sub_cls= 4 (p=0.096), obj_cls=26 (p=0.065), sub_box=[0.500,0.500,0.119,0.119], obj_box=[0.500,0.500,0.119,0.119]
         verbs: [(92, 0.05522904917597771), (128, 0.04234924912452698), (53, 0.03920532763004303)]
  Query   2: sub_cls= 4 (p=0.096), obj_cls=26 (p=0.065), sub_box=[0.500,0.500,0.119,0.119], obj_box=[0.500,0.500,0.119,0.119]
         verbs: [(92, 0.05531289055943489), (128, 0.04235837981104851), (53, 0.039321836084127426)]
  Query   3: sub_cls= 4 (p=0.096), obj_cls=26 (p=0.065), sub_box=[0.500,0.500,0.119,0.119], obj_box=[0.500,0.500,0.119,0.119]
         verbs: [(92, 0.055121153593063354), (128, 0.042352065443992615), (53, 0.039120614528656006)]
  Query   4: sub_cls= 

## 8. Summary: The Complete VRDFormer Training Pipeline

```
                    VRDFORMER TRAINING PIPELINE
                    ===========================

DATA PREPARATION:
  Raw annotations (JSON) --prepare.py--> metadata/<db>_annotations.pkl
                                     + metadata/<db>_train_frames_stage{1,2}.json

STAGE 1 (Detection + Tracking):
  ├── Dataset: Frame pairs (current + previous)
  ├── Dataset output: (img [3,H,W], target [sub/obj boxes+labels+track_ids+prev_frame])
  ├── DataLoader output: (samples [B,3,H,W], targets [B target dicts])
  ├── Model forward:
  │   1. Run prev frame through model (no_grad) -> extract track_query_hs_embeds
  │   2. Run current frame: backbone -> encoder -> decoder (with track+static queries)
  │   3. Prediction heads -> sub/obj/verb logits + sub/obj boxes
  ├── Loss: Hungarian match -> focal cls + verb focal + L1 + GIoU
  ├── Optimizer: AdamW with diff LRs (backbone 1e-5, rest 5e-5)
  ├── Output: frame-level subject-object pair detections
  └── Epochs: 7 (typically)

STAGE 2 (Relation Classification):
  ├── Dataset: 8-frame clips
  ├── Dataset output: (clip [T,3,H,W], [T target dicts])
  ├── DataLoader output: (samples [T,3,H,W], targets [1][T dicts])
  ├── Model forward (per frame):
  │   1. Backbone + encoder
  │   2. ROI Align at GT boxes -> s_embed, o_embed
  │   3. so_linear(cat(s,o)) -> query init
  │   4. Decoder -> rel_embed
  │   5. memory_update(so_track_id, rel_embed)
  ├── At EOS: relation_classifier(memory) -> mean_pool -> classify
  ├── Loss: focal classification (sub/obj/verb) - no box loss
  ├── Batch size: 1
  ├── Output: per-tracklet relation predictions
  └── Epochs: 3 (typically)

EVALUATION:
  ├── Stage 2 model, eval mode
  ├── Process all frames in val video
  ├── relation_classifier -> argmax over verb logits per GT triplet
  ├── evaluate(groundtruth, prediction, dataset)
  ├── Metrics: detection mAP, rec@50/100, tagging pre@1/5/10
  └── Overall + zero-shot + generalized zero-shot settings
```

### Quick Reference: Tensor Shapes

| Stage | Data Input | Model Input | Model Output (per query) |
|-------|-----------|-------------|--------------------------|
| 1 | Frame pair (3,H,W)×2 | (B,3,H,W) NestedTensor | sub_cls(36), obj_cls(36), verb_cls(132), sub_box(4), obj_box(4) |
| 2 | 8-frame clip (8,3,H,W) | (1,3,H,W) per frame | rel_embed(256) accumulated, classified at EOS |

### Where to Go From Here

- Read the paper: "VRDFormer: End-to-End Video Visual Relation Detection with Transformers" (CVPR 2022)
- Experiment with different configs in `configs/`
- Try the deformable variant (requires compiled `models/ops`)
- Use `scripts/` for full distributed training on multi-GPU
- Check `docs/DATA.md` and `docs/INSTALL.md` for setup details